# Notebook setup

Run the next cell first. In Google Colab it clones the course repository, then imports the libraries. If you cloned the repository and run the notebook locally (see the README), it only does the imports.

In [1]:
#@title Setup: clone the course repository (Google Colab only) and import libraries
import os
import sys

# In Google Colab the repository is cloned and the notebook moves into its "scripts" folder.
if "google.colab" in sys.modules and not os.path.isdir("modules"):
    if not os.path.isdir("analytics-course-scripts"):
        !git clone -q https://github.com/DataHow/analytics-course-scripts.git
    os.chdir("analytics-course-scripts/scripts")
# When the notebook is started from the repository root, move into "scripts"
if os.path.isdir("scripts/modules"):
    os.chdir("scripts")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import copy
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Gaussian Processes

In this notebook we investigate the main idea behind Gaussian processes (GPs) and how conditional probability can be used to make predictions from data.

The idea is that there is a function that is unknown to us (the "originator function" below). We sample the function in different "experiments" and get noisy data in return. We use such data to construct a Gaussian process.


## Create the originator function

Here we first create an originator function that is nonlinear and has modifiable parameters. The function is sampled at the points `XPOINTS`.


In [2]:
""" Observation points """
XPOINTS = np.array(range(1,7))
""" Function parameters """
beta0 = 6
beta1 = -3
beta2 = 0.5
sigma = 0.2
""" Function randomness """
SEED = 42
np.random.seed(SEED)
""" Originator function definition """
def f(x, beta0=6, beta1=-3, beta2=0.5, sigma=0.2):
    return np.matmul(np.vstack((np.ones(len(x)), np.array(x), np.array(x) **2)).T,np.array([beta0,beta1,beta2])) + sigma*np.random.normal(0,1,len(x))

In [3]:
#@title Plot originator function with generated data
XGRID = np.arange(0,7,0.1)
fig = px.line(x=XGRID,y=f(XGRID,beta0,beta1,beta2,sigma=0),color=px.Constant("Originator function"), title = "Generated function we try to learn",labels=dict(x="x", y="y", color="Legend"))
fig.add_trace(go.Scatter(x=XPOINTS,y=f(XPOINTS,beta0,beta1,beta2),mode='markers',marker=dict(size=10),name="Generated data"))
fig.show()

### Generate data from the function

Now we generate data from that function. `NO_EXP` defines the number of experiments, i.e. how many times the underlying process is sampled at the different values of x. To generate the samples we slightly vary the underlying parameters of the function for each experiment, so every experiment is a slightly different realization of the process.


In [4]:
""" Number of experiments """
NO_EXP = 25

In [5]:
#@title Generate and plot the experiments
# Plot generated experiments
np.random.seed(SEED)
YEXP = np.zeros((NO_EXP,len(XPOINTS)))
fig = px.scatter()
for n in range(NO_EXP):
    YEXP[n,:] = f(XPOINTS,(np.random.normal()*0.5+beta0),(np.random.normal()*0.1+beta1),(np.random.normal()*0.01+beta2))
    fig.add_trace(go.Scatter(x=XPOINTS,y=YEXP[n,:],mode='lines+markers',line=dict(color = 'rgb(49,130,189)')))
fig.update_layout(showlegend=False,title="Generated data",xaxis_title="x",yaxis_title="y")
fig.update_layout(width=1000)
fig.show()


## Conditional probability of the next step


### Plot correlations

Here we show as an example how the data at x = 1 are correlated to those at x = 2. Below we compute the mean of the function at each sampling point and the corresponding covariance and correlation matrices.


In [6]:
""" Select steps to plot on axis """
STEP_XAXIS = 1
STEP_YAXIS = 2

In [7]:
#@title Plot correlation between selected steps
fig = px.scatter(x=YEXP[:,STEP_XAXIS-1],y=YEXP[:,STEP_YAXIS-1], 
                 title = "Correlation between step x = "+str(STEP_XAXIS)+" and x = "+str(STEP_YAXIS)+" with correlation being "+str(np.round(np.corrcoef(YEXP[:,STEP_XAXIS-1],YEXP[:,STEP_YAXIS-1])[0,1],3)),
                 labels=dict(x="Y(x = "+str(STEP_XAXIS)+")", y="Y(x = "+str(STEP_YAXIS)+")"),
                 marginal_x="box", marginal_y="box",)
fig.update_layout(width=1000,height=700)
fig.show()

Here we compute and visualize the mean, covariance, and correlation for all the steps.

In [8]:
Y_mu = np.mean(YEXP,axis=0)
Y_cov = np.cov(YEXP,rowvar=False)
Y_corr = np.corrcoef(YEXP,rowvar=False)
print('Mean: \n',pd.DataFrame(Y_mu,index=XPOINTS).T)
print('\nCovariance: \n',pd.DataFrame(Y_cov,index=XPOINTS,columns=XPOINTS))
print('\nCorrelation: \n',pd.DataFrame(Y_corr,index=XPOINTS,columns=XPOINTS))

Mean: 
           1         2         3        4         5         6
0  3.513084  1.966933  1.477335  1.93705  3.367365  5.907705

Covariance: 
           1         2         3         4         5         6
1  0.194905  0.179594  0.193249  0.198866  0.149583  0.153221
2  0.179594  0.246583  0.226285  0.232950  0.206985  0.213588
3  0.193249  0.226285  0.314380  0.278909  0.287660  0.326630
4  0.198866  0.232950  0.278909  0.342012  0.318072  0.396732
5  0.149583  0.206985  0.287660  0.318072  0.384331  0.460696
6  0.153221  0.213588  0.326630  0.396732  0.460696  0.672466

Correlation: 
           1         2         3         4         5         6
1  1.000000  0.819216  0.780691  0.770245  0.546535  0.423225
2  0.819216  1.000000  0.812732  0.802160  0.672364  0.524518
3  0.780691  0.812732  1.000000  0.850579  0.827558  0.710385
4  0.770245  0.802160  0.850579  1.000000  0.877308  0.827260
5  0.546535  0.672364  0.827558  0.877308  1.000000  0.906204
6  0.423225  0.524518  0.710385  

In [9]:
# Assume a multivariate normal distribution for the data with the mean and covariance computed above
# and draw a large number of samples from it
NUM_MVN_SAMPLES = 500000
YMN = np.random.multivariate_normal(Y_mu, Y_cov, NUM_MVN_SAMPLES)

### Using the correlations

If we define the correlation structure and suppose that the underlying probability density function is a multivariate normal distribution, we can sample from this distribution and generate new data. Below we plot pairs of sampled data at x = 1 and x = 2.


In [10]:
""" Define number of samples to generate from multivariate distribution """
NUM_SAMPLES = 50
""" Select steps to plot on axis """
STEP_XAXIS = 1
STEP_YAXIS = 2

In [11]:
#@title Plot samples from the multivariate normal distribution
fig = px.scatter()
for n in range(NUM_SAMPLES):
    fig.add_trace(go.Scatter(x=[STEP_XAXIS,STEP_YAXIS],y=[YMN[n,STEP_XAXIS-1],YMN[n,STEP_YAXIS-1]],mode='lines+markers',line=dict(color = 'rgb(49,130,189)')))
fig.update_xaxes(range=[STEP_XAXIS-1,STEP_YAXIS+1])
fig.update_layout(showlegend=False,title="Generated data",xaxis_title="x",yaxis_title="y")
fig.update_layout(width=1000,height=500)
fig.show()


## Conditional probability given a single observation

What is the distribution of $f(x)$ at $x = 2$ if we know that $f(x = 1) = 3$?
This is a typical conditional probability problem, where we have to "conditionally" sample from the original distribution: we keep only the samples whose value at $x = 1$ lies within a small tolerance around the observed value.

Hence we look at the conditional probability $P(B \mid A)$ where $B: x =$ `STEP_B` and $A: f(x =$ `STEP_A` $) =$ `VALUE_A`.


In [12]:
""" Step at which we want to find out conditional probability """
STEP_B = 2
""" Step & Value at which we condition on / we have observed """
STEP_A = 1
VALUE_A= 3
""" tolerance region for observed value range """
TOL = 0.02

In [13]:
# Keep only the samples of the distribution that match the observation (within the tolerance)
rows = np.logical_and(YMN[:,STEP_A-1] > VALUE_A - TOL, YMN[:,STEP_A-1] < VALUE_A + TOL)
print('Number of matching samples: ', rows.sum())
# Compute the conditional mean and standard deviation from the matching samples
mu_step = np.mean(YMN[rows,STEP_B-1])
std_step = np.std(YMN[rows,STEP_B-1])
print('Conditional Mean: ', round(mu_step, 4))
print('Conditional Deviation: ', round(std_step, 4))


Number of matching samples:  9141
Conditional Mean:  1.4993
Conditional Deviation:  0.284


In [14]:
#@title Plot the histogram of the conditional distribution
# Plot histogram
fig = px.histogram(YMN[rows,STEP_B-1],histnorm='probability density')
fig.update_layout(bargap=0.01,showlegend=False,title="Conditional Probability",xaxis_title="P(x="+str(STEP_B)+"|f(x="+str(STEP_A)+")="+str(VALUE_A)+")",yaxis_title="Probability density")
fig.add_vline(mu_step)
fig.add_vline(mu_step-std_step,line=dict(color='orange'))
fig.add_vline(mu_step+std_step,line=dict(color='orange'))
fig.add_vline(mu_step-2*std_step,line=dict(color='orange',dash='dash'))
fig.add_vline(mu_step+2*std_step,line=dict(color='orange',dash='dash'))
fig.show()


Hence we can show the conditional probability of the function given our observation: the lines are samples that pass through the observed point, the orange marker and bar show the conditional mean and the 2-sigma interval.

In [15]:
""" Select number of simulations to plot """
NUM_SIMULATIONS = 20

In [16]:
#@title Plot conditional samples
fig = px.scatter()
for n in range(NUM_SIMULATIONS):
    fig.add_trace(go.Scatter(x=[STEP_A,STEP_B],y=[VALUE_A,YMN[rows,STEP_B-1][n]],mode='lines+markers',line=dict(color = 'rgb(49,130,189)')))
fig.add_trace(go.Scatter(x=[STEP_B,STEP_B],y=[mu_step-2*std_step,mu_step+2*std_step],mode='lines+markers',line=dict(color = px.colors.qualitative.G10[1])))
fig.add_trace(go.Scatter(x=[STEP_B,STEP_B],y=[mu_step,mu_step],mode='markers',marker=dict(size=10,color = px.colors.qualitative.G10[1])))
fig.add_trace(go.Scatter(x=[STEP_A,STEP_A],y=[VALUE_A,VALUE_A],mode='markers',marker=dict(size=15,color = 'rgb(49,190,130)')))
fig.update_layout(showlegend=False,title="Conditional Probability given observations",xaxis_title="x",yaxis_title="y")
fig.update_xaxes(range=[STEP_A-1,STEP_B+1])
fig.show()

## Conditional probability given an observation, for all steps


### Histogram of f(x) at x = 2, 3, 4, 5, 6 given f(x = 1) = 3

It is possible to repeat the exercise above, where we fixed the value of $f(x = 1) = 3$. This time we check the probability of measuring $f(x)$ at $x = 2, 3, 4, 5, 6$.
As shown in the correlation matrix above, the values at these positions are correlated to those at $x = 1$, although the correlation decreases further away from $x = 1$, and so the corresponding uncertainty increases.


In [17]:
""" Step & Value at which we condition on / we have observed """
STEP_A = 1
VALUE_A= 3
""" tolerance region for observed value range """
TOL = 0.02
""" Compute conditional probabilities for remaining steps """
STEPS_OTHER = [2,3,4,5,6]


In [18]:
STEPS_ALL = copy.copy(STEPS_OTHER)
STEPS_ALL.insert(0,STEP_A)
# Keep only the samples that match the observation
rows = np.logical_and(YMN[:,STEP_A-1] > VALUE_A - TOL, YMN[:,STEP_A-1] < VALUE_A + TOL)
print('Number of matching samples: ', rows.sum())
mu_OTHER = []
std_OTHER = []
for i in STEPS_OTHER:
    mu_OTHER.append(np.mean(YMN[rows,i-1]))
    std_OTHER.append(np.std(YMN[rows,i-1]))
df_sum = pd.DataFrame(columns = ["steps","means"])
df_sum["steps"]= STEPS_OTHER
df_sum["means"]= mu_OTHER
print('\nMeans: \n',df_sum.to_string(index=False))
Y_corr = np.corrcoef(YMN[rows][:,[i-1 for i in STEPS_OTHER]],rowvar=False)
print('\nCorrelations: \n',pd.DataFrame(Y_corr,columns=STEPS_OTHER,index=STEPS_OTHER),'\n')


Number of matching samples:  9141

Means: 
  steps    means
     2 1.499287
     3 0.973808
     4 1.415779
     5 2.978939
     6 5.504204

Correlations: 
           2         3         4         5         6
2  1.000000  0.486872  0.471913  0.468463  0.347173
3  0.486872  1.000000  0.630219  0.768298  0.681114
4  0.471913  0.630219  1.000000  0.853663  0.867466
5  0.468463  0.768298  0.853663  1.000000  0.893380
6  0.347173  0.681114  0.867466  0.893380  1.000000 



In [19]:
#@title Plot the histograms of the conditional distributions
fig = go.Figure()
for i in STEPS_OTHER:
    fig.add_trace(go.Histogram(x=YMN[rows,i-1],histnorm='probability density',name="x = "+str(i)))
# Overlay both histograms
fig.update_layout(barmode='overlay')
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.update_layout(bargap=0.01,showlegend=True,title="Conditional Probability at all steps",xaxis_title="P(x=" +str(STEPS_OTHER)+" | f(x="+str(STEP_A)+")="+str(VALUE_A)+")",yaxis_title="Probability density")
fig.show()

### Visualizing the corresponding function realizations


In [20]:
""" Select number of simulations to plot """
NUM_SIMULATIONS = 40

In [21]:
#@title Plot conditional samples
fig = px.scatter()
for n in range(NUM_SIMULATIONS):
    fig.add_trace(go.Scatter(x=STEPS_ALL,y=np.concatenate((np.array(VALUE_A),YMN[rows][:,[i-1 for i in STEPS_OTHER]][n]),axis=None),mode='lines+markers',line=dict(color = 'rgb(49,130,189)')))
fig.add_trace(go.Scatter(x=[STEP_A,STEP_A],y=[VALUE_A,VALUE_A],mode='markers',marker=dict(size=15,color = 'rgb(49,190,130)')))
for i,s in enumerate(STEPS_OTHER):
    fig.add_trace(go.Scatter(x=[s,s],y=[mu_OTHER[i]],mode='markers',marker=dict(size=10,color = px.colors.qualitative.G10[1])))
    fig.add_trace(go.Scatter(x=[s,s],y=[mu_OTHER[i]-2*std_OTHER[i],mu_OTHER[i]+2*std_OTHER[i]],mode='lines+markers',line=dict(color = px.colors.qualitative.G10[1])))
fig.update_layout(showlegend=False,title="Conditional Probability given observations",xaxis_title="x",yaxis_title="y")
fig.update_xaxes(range=[STEPS_ALL[0]-1,STEPS_ALL[-1]+1])
fig.show()

## Conditional probability given multiple observations

Let's suppose we fix the starting point at $f(x = 1) = 3$ and additionally observe $f(x = 4) = 2$. What is the corresponding distribution of the remaining values?


In [22]:
""" Step & Value at which we condition on / we have observed """
STEP_A = 1
VALUE_A= 3
""" Additional Step & Value at which we condition on / we have observed """
STEP_B = 4
VALUE_B= 2
""" tolerance region for observed value range """
TOL = 0.02
""" Compute conditional probabilities for remaining steps """
STEPS_OTHER = [2,3,5,6]

In [23]:
STEPS_ALL = copy.copy(STEPS_OTHER)
STEPS_ALL.append(STEP_A)
STEPS_ALL.append(STEP_B)
STEPS_ALL.sort()
# Keep only the samples that match both observations
rows = np.logical_and(np.logical_and(YMN[:,STEP_A-1] > VALUE_A - TOL, YMN[:,STEP_A-1] < VALUE_A + TOL),
                      np.logical_and(YMN[:,STEP_B-1] > VALUE_B - TOL, YMN[:,STEP_B-1] < VALUE_B + TOL))
print('Number of matching samples: ', rows.sum())
mu_OTHER = []
std_OTHER = []
for i in STEPS_OTHER:
    mu_OTHER.append(np.mean(YMN[rows,i-1]))
    std_OTHER.append(np.std(YMN[rows,i-1]))
df_sum = pd.DataFrame(columns = ["steps","means"])
df_sum["steps"]= STEPS_OTHER
df_sum["means"]= mu_OTHER
print('\nMeans: \n',df_sum.to_string(index=False))
Y_corr = np.corrcoef(YMN[rows][:,[i-1 for i in STEPS_OTHER]],rowvar=False)
print('\nCorrelations: \n',pd.DataFrame(Y_corr,columns=STEPS_OTHER,index=STEPS_OTHER),'\n')


Number of matching samples:  135

Means: 
  steps    means
     2 1.731107
     3 1.307180
     5 3.662776
     6 6.484117

Correlations: 
           2         3         5         6
2  1.000000  0.186237  0.120908 -0.101213
3  0.186237  1.000000  0.620883  0.362417
5  0.120908  0.620883  1.000000  0.636384
6 -0.101213  0.362417  0.636384  1.000000 



In [24]:
#@title Plot the histograms of the conditional distributions
fig = go.Figure()
for i in STEPS_OTHER:
    fig.add_trace(go.Histogram(x=YMN[rows,i-1],histnorm='probability density',name="x = "+str(i)))
# Overlay both histograms
fig.update_layout(barmode='overlay')
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.update_layout(bargap=0.01,showlegend=True,title="Conditional Probability at all steps",xaxis_title="P(x=" +str(STEPS_OTHER)+" | f(x="+str(STEP_A)+")="+str(VALUE_A)+"), f(x="+str(STEP_B)+")="+str(VALUE_B)+")",yaxis_title="Probability density")
fig.show()

In [25]:
""" Select number of simulations to plot """
NUM_SIMULATIONS = 40

In [26]:
#@title Plot conditional samples
fig = px.scatter()
for n in range(NUM_SIMULATIONS):
    # assemble the sampled values in the order of STEPS_ALL (observed values are fixed)
    sample = dict(zip(STEPS_OTHER, YMN[rows][:,[i-1 for i in STEPS_OTHER]][n]))
    sample[STEP_A] = VALUE_A
    sample[STEP_B] = VALUE_B
    values_all = [sample[s] for s in STEPS_ALL]
    fig.add_trace(go.Scatter(x=STEPS_ALL,y=values_all,mode='lines+markers',line=dict(color = 'rgb(49,130,189)')))
fig.add_trace(go.Scatter(x=[STEP_A,STEP_A],y=[VALUE_A,VALUE_A],mode='markers',marker=dict(size=15,color = 'rgb(49,190,130)')))
fig.add_trace(go.Scatter(x=[STEP_B,STEP_B],y=[VALUE_B,VALUE_B],mode='markers',marker=dict(size=15,color = 'rgb(49,190,130)')))
for i,s in enumerate(STEPS_OTHER):
    fig.add_trace(go.Scatter(x=[s,s],y=[mu_OTHER[i]],mode='markers',marker=dict(size=10,color = px.colors.qualitative.G10[1])))
    fig.add_trace(go.Scatter(x=[s,s],y=[mu_OTHER[i]-2*std_OTHER[i],mu_OTHER[i]+2*std_OTHER[i]],mode='lines+markers',line=dict(color = px.colors.qualitative.G10[1])))
fig.update_layout(showlegend=False,title="Conditional Probability given observations",xaxis_title="x",yaxis_title="y")
fig.update_xaxes(range=[STEPS_ALL[0]-1,STEPS_ALL[-1]+1])
fig.show()

# Kernels in Gaussian processes

In this section we try to learn an unknown function (the "originator function" below) from a few samples.

The key point is the definition of a kernel function, which defines the similarity between data points. The key parameter is `sigma`, which defines the so-called similarity scale (length scale).

In the following we show how these simple concepts can be put together to predict the value of the unknown function away from the data, and how the Gaussian process model behaves in response to changes in the model parameters.


## Create the originator function

As above, we define a function which we try to learn. This time it is a cubic polynomial sampled without noise at three points.


In [27]:
""" Define observations steps """
XPOINTS = np.array((2,5,8))
""" Fix seed """
np.random.seed(42)
""" Function parameters """
beta0 = 6
beta1 = -3
beta2 = 0.5
beta3 = -0.025
eps = 0
# Originator function
def f(x, beta0=6, beta1=-3, beta2=0.5, beta3=-0.025,eps=0):
    return np.matmul(np.vstack((np.ones(len(x)), np.array(x), np.array(x) **2,np.array(x) **3)).T,np.array([beta0,beta1,beta2,beta3])) + eps*np.random.normal(0,1,len(x))

In [28]:
#@title Sample the function and plot it
# Obtain measurements
YPOINTS=f(XPOINTS,beta0,beta1,beta2,beta3,eps)
# Plot originator function with generated data
xgrid = np.arange(0,12,0.1)
fig = go.Figure()
fig.add_trace(go.Scatter(x=xgrid,y=f(xgrid),mode='lines',name="Originator function"))
fig.add_trace(go.Scatter(x=XPOINTS,y=YPOINTS,mode='markers',marker=dict(size=10),name="Generated data"))
fig.update_layout(showlegend=True,title="Generating function we try to learn",xaxis_title="x",yaxis_title="y")
fig.show()

### Create a kernel function

You can select different values of the similarity scale `sigma`. Three kernels are prepared in the function below (uncomment the one you want to try).
For better understanding, we plot the value of the similarity (kernel) function centred on the first observation (i.e. the kernel is $= 1$ at x = 2).


In [29]:
""" Select point around which we plot kernel """
PLOT_BASE = XPOINTS[0] # x=2
""" Kernel parameters """
sigma = 2 
def kernel_fcn(xi,xj,sigma=2): 
    ## This is rbf / normal distribution kernel 
    return np.exp(-0.5*((np.array(xi)-np.array(xj))/sigma)**2) 
    ## This is laplacian kernel
    # return np.exp(- sigma*np.abs((np.array(xi) - np.array(xj)))) 
    ## This is polynomial kernel
    # return (np.array(xi)*np.array(xj)+1)**sigma 

In [30]:
#@title Plot the originator function with the kernel
# Plot originator function with generated data
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=xgrid,y=f(xgrid),mode='lines',name="Originator function"))
fig.add_trace(go.Scatter(x=xgrid,y=kernel_fcn(xgrid,PLOT_BASE,sigma),mode='lines',marker=dict(size=10),name="Kernel function"),secondary_y=True)
fig.add_trace(go.Scatter(x=XPOINTS,y=YPOINTS,mode='markers',marker=dict(size=15),name="Generated data"))
fig.update_layout(showlegend=True,title="Generating function with kernel",xaxis_title="x",yaxis_title="y")
fig.update_yaxes(title_text="Kernel function", secondary_y=True)
fig.show()

**Compute the kernel matrix** between all pairs of observation points.

In [31]:
no_data = len(XPOINTS)
k = np.zeros((no_data,no_data))
for i in range(no_data):
    k[i,:] = kernel_fcn(XPOINTS[i],XPOINTS,sigma)
K = pd.DataFrame(k)
K.index = XPOINTS
K.columns = XPOINTS
print("Kernel given observations: \n",K)

Kernel given observations: 
           2         5         8
2  1.000000  0.324652  0.011109
5  0.324652  1.000000  0.324652
8  0.011109  0.324652  1.000000


### Add a new observation point

You can define a new point `XPOINTS_NEW` at which we want to predict the function.
The new point is added in the last position of the new kernel matrix, which is now 4x4, and calculated below.


In [32]:
""" Define a new observation point """
XPOINTS_NEW = 6

In [33]:
# Update kernel
x_pred = np.array(XPOINTS_NEW)
x_new = np.append(XPOINTS,x_pred)
k_pred = np.zeros((len(XPOINTS)+1,len(XPOINTS)+1))
# Copy old kernel
k_pred[0:len(XPOINTS),0:len(XPOINTS)] = k
# Compute kernel for new observations
k_pred[:,len(XPOINTS)] =  kernel_fcn(x_pred,x_new,sigma)
k_pred[len(XPOINTS),:] =  kernel_fcn(x_pred,x_new,sigma)
# Visualize kernel
K_pred = pd.DataFrame(k_pred); K_pred.index = x_new ; K_pred.columns = x_new
print("\nKernel given new observations: \n",K_pred)


Kernel given new observations: 
           2         5         8         6
2  1.000000  0.324652  0.011109  0.135335
5  0.324652  1.000000  0.324652  0.882497
8  0.011109  0.324652  1.000000  0.606531
6  0.135335  0.882497  0.606531  1.000000


### Create the covariance matrix

A second parameter is needed: the prior variance of the function values `SIGMA_F2`. It scales the kernel (which is 1 on the diagonal) to the covariance matrix and expresses our prior knowledge about how much the function varies.


In [34]:
""" Prior variance of the function values """
SIGMA_F2 = 2.0
cov_pred = SIGMA_F2*k_pred
Cov_pred = pd.DataFrame(cov_pred); Cov_pred.index = x_new ; Cov_pred.columns = x_new
print("\n Covariance between points: \n",Cov_pred)


 Covariance between points: 
           2         5         8         6
2  2.000000  0.649305  0.022218  0.270671
5  0.649305  2.000000  0.649305  1.764994
8  0.022218  0.649305  2.000000  1.213061
6  0.270671  1.764994  1.213061  2.000000


### New point prediction

In Gaussian processes, the prediction at a new point $x^*$ has the target value $f^* = f(x^*)$, which is equal to $y^*$ in the noise-free case we are dealing with now. The case with measurement error is described below.

Given the training and test points defined above, the **joint distribution** of the Gaussian process function values over the training and test points ($f$ and $f^*$ respectively) is

\begin{align}\begin{bmatrix} f \\ f^* \end{bmatrix} \sim \mathcal{N} \left( 0,\begin{bmatrix} K(x,x) & K(x,x^*)\\ K(x^*,x) & K(x^*,x^*) \end{bmatrix} \right)\end{align}


However, when making a new prediction, what we are actually interested in (as visualized above) is the conditional distribution of the possible predictions $f^*$ given our train and test data and train targets $x,x^*,f$. This is given by:

\begin{align} f^* | x,x^*,f \sim \mathcal{N}\left(\bar{f}(x^*),Cov(\bar{f}(x^*))\right)\end{align}

where the mean is:
\begin{align} \bar{f}(x^*) = K(x^*,x) K(x,x)^{-1} f\end{align}

and the covariance is:
\begin{align} Cov(\bar{f}(x^*)) = K(x^*,x^*) - K(x^*,x) K(x,x)^{-1} K(x,x^*)\end{align}

### Create the conditional multivariate distribution

The function below implements the two formulas above: it computes the conditional mean and variance at the new point, given the other observations. The data are de-meaned before conditioning, because the GP prior has zero mean.


In [35]:
def mvncond2(joint_mean, joint_cov, index, obs_diff):
    # joint_mean: mean vector of the joint distribution (zero vector)
    # joint_cov:  covariance matrix including the new point
    # index:      position of the new point in the covariance matrix
    # obs_diff:   obs_f - obs_mu, the de-meaned observations
    i = index
    kxx = joint_cov[0:i,0:i]      # K(x,x)
    kxsx = joint_cov[i,0:i]       # K(x*,x)
    s12s22 = np.matmul(kxsx,np.linalg.inv(kxx))
    cond_mean = joint_mean[i] + np.matmul(s12s22,(obs_diff-joint_mean[0:i]))
    cond_cov = joint_cov[i,i] - np.matmul(s12s22,np.transpose(kxsx))
    return cond_mean, cond_cov


In [36]:
# Calculate the conditional mean and std at the new point
joint_mean = np.zeros(len(XPOINTS)+1)
joint_cov = cov_pred
obs_f = YPOINTS
obs_mu = np.mean(YPOINTS)
cond_mean,cond_cov = mvncond2(joint_mean,joint_cov,len(XPOINTS),obs_f - obs_mu)
new_obs_mu = cond_mean + obs_mu
print('Conditional mean of the new point: ', round(new_obs_mu,4))
new_obs_std = np.sqrt(cond_cov)
print('Conditional std of the new point:  ', round(new_obs_std,4))


Conditional mean of the new point:  0.4507
Conditional std of the new point:   0.4269


In [37]:
#@title Plot the prediction at the new point
# Plot originator function with generated data
fig = go.Figure()
fig.add_trace(go.Scatter(x=xgrid,y=f(xgrid),mode='lines',name="Originator function"))
fig.add_trace(go.Scatter(x=XPOINTS,y=YPOINTS,mode='markers',marker=dict(size=10),name="Generated data"))
fig.add_trace(go.Scatter(x=[XPOINTS_NEW,XPOINTS_NEW],y=[new_obs_mu-2*new_obs_std,new_obs_mu+2*new_obs_std],mode='markers+lines',line=dict(color = px.colors.qualitative.G10[3]),name = "Conditional Deviation"))
fig.add_trace(go.Scatter(x=[XPOINTS_NEW,XPOINTS_NEW],y=[new_obs_mu,new_obs_mu],mode='markers',marker=dict(size=10,color = px.colors.qualitative.G10[3]),name = "Conditional Mean"))
fig.update_layout(showlegend=True,title="Conditional prediction at a new point x="+str(XPOINTS_NEW),xaxis_title="x",yaxis_title="y")
fig.show()

### Compute predictions for all x

Here we repeat exactly the same procedure as above for all the points of the x-axis grid. This gives the GP mean prediction and its uncertainty band.


In [38]:
# Loop through all points and compute the conditional mean and std
xgrid_mu = []
xgrid_std = []
for new_obs_point in xgrid:
    # Update kernel
    x_pred = np.array(new_obs_point)
    x_new = np.append(XPOINTS,x_pred)
    k_pred = np.zeros((len(XPOINTS)+1,len(XPOINTS)+1))
    # Copy old kernel
    k_pred[0:len(XPOINTS),0:len(XPOINTS)] = k
    # Compute kernel for new observations
    k_pred[:,len(XPOINTS)] =  kernel_fcn(x_pred,x_new,sigma)
    k_pred[len(XPOINTS),:] =  kernel_fcn(x_pred,x_new,sigma)
    # Compute joint distribution
    joint_mean = np.zeros(len(XPOINTS)+1)
    joint_cov = SIGMA_F2*k_pred
    obs_f = YPOINTS
    obs_mu = np.mean(YPOINTS)
    # Compute conditional distribution
    cond_mean,cond_cov = mvncond2(joint_mean,joint_cov,len(XPOINTS),obs_f - obs_mu)
    xgrid_mu.append(cond_mean + obs_mu)
    xgrid_std.append(np.sqrt(cond_cov))


In [39]:
#@title Plot the prediction for all points
# Plot the distribution for all points
fig = go.Figure()
fig.add_trace(go.Scatter(x=xgrid,y=f(xgrid),mode='lines',name="Originator function"))
fig.add_trace(go.Scatter(x=XPOINTS,y=YPOINTS,mode='markers',marker=dict(size=10),name="Generated data"))
fig.add_trace(go.Scatter(x=xgrid,y=xgrid_mu,mode='lines',line=dict(width=3,color = px.colors.qualitative.G10[3]),name = "Conditional Mean"))
fig.add_trace(go.Scatter(x=xgrid,y=np.array(xgrid_mu)-2*np.array(xgrid_std),mode='lines',line=dict(width=1,color = px.colors.qualitative.G10[3]),showlegend=False))
fig.add_trace(go.Scatter(x=xgrid,y=np.array(xgrid_mu)+2*np.array(xgrid_std),mode='lines',fill='tonexty',line=dict(width=1,color = px.colors.qualitative.G10[3]),name = "Conditional Deviation"))

fig.update_layout(showlegend=True,title="Conditional prediction at all points",xaxis_title="x",yaxis_title="y")
fig.show()

### Effect of measurement error

In this section we simulate the effect of measurement error on the model. We sample the function twice at 4 regular points and add noise to the measurements.


In [40]:
""" Generate observations """
XPOINTS = np.arange(2,12,3)
""" Specify number of measurements at the observation points and noise """
NUM_MEASUREMENTS = 2
NOISE_MEASUREMENTS = 0.2
x = np.repeat(XPOINTS,NUM_MEASUREMENTS)
""" Fix seed """
np.random.seed(43)

In [41]:
#@title Sample the function with measurement noise and plot it
# Predict values with measurement noise
eps = NOISE_MEASUREMENTS
y = f(x,eps=eps)
# Plot originator function with generated data
xgrid = np.arange(0,12,0.1)
fig = go.Figure()
fig.add_trace(go.Scatter(x=xgrid,y=f(xgrid),mode='lines',name="Originator function"))
fig.add_trace(go.Scatter(x=x,y=y,mode='markers',marker=dict(size=10),name="Data with measurement error"))
fig.update_layout(showlegend=True,title="Generating function we try to learn",xaxis_title="x",yaxis_title="y")
fig.show()

**We compute the kernel** between all pairs of (noisy) observation points.

In [42]:
no_data = len(x)
k = np.zeros((no_data,no_data))
for i in range(no_data):
    k[i,:] = kernel_fcn(x[i],x,sigma)
K = pd.DataFrame(k)
K.index = x
K.columns = x
print("Kernel given observations: \n",K)

Kernel given observations: 
           2         2         5         5         8         8         11  \
2   1.000000  1.000000  0.324652  0.324652  0.011109  0.011109  0.000040   
2   1.000000  1.000000  0.324652  0.324652  0.011109  0.011109  0.000040   
5   0.324652  0.324652  1.000000  1.000000  0.324652  0.324652  0.011109   
5   0.324652  0.324652  1.000000  1.000000  0.324652  0.324652  0.011109   
8   0.011109  0.011109  0.324652  0.324652  1.000000  1.000000  0.324652   
8   0.011109  0.011109  0.324652  0.324652  1.000000  1.000000  0.324652   
11  0.000040  0.000040  0.011109  0.011109  0.324652  0.324652  1.000000   
11  0.000040  0.000040  0.011109  0.011109  0.324652  0.324652  1.000000   

          11  
2   0.000040  
2   0.000040  
5   0.011109  
5   0.011109  
8   0.324652  
8   0.324652  
11  1.000000  
11  1.000000  


And create predictions. The measurement noise variance is added to the diagonal of the covariance matrix, which tells the GP that the observations are not exact.

In [43]:
# Loop through all points and compute the conditional mean and std
xgrid_mu = []
xgrid_std = []
for new_obs_point in xgrid:
    num_data = len(x)
    # Update kernel
    x_pred = np.array(new_obs_point)
    x_new = np.append(x,x_pred)
    k_pred = np.zeros((num_data+1,num_data+1))
    # Copy old kernel
    k_pred[0:num_data,0:num_data] = k
    # Compute kernel for new observations
    k_pred[:,num_data] =  kernel_fcn(x_pred,x_new,sigma)
    k_pred[num_data,:] =  kernel_fcn(x_pred,x_new,sigma)
    # Compute joint distribution
    joint_mean = np.zeros(num_data+1)
    # IMPORTANT! we add the measurement noise variance to the diagonal of the covariance matrix
    joint_cov = SIGMA_F2*k_pred + eps**2 * np.eye(num_data+1)
    obs_f = y
    obs_mu = np.mean(y)
    # Compute conditional distribution
    cond_mean,cond_cov = mvncond2(joint_mean,joint_cov,num_data,obs_f - obs_mu)
    xgrid_mu.append(cond_mean + obs_mu)
    xgrid_std.append(np.sqrt(cond_cov))


In [44]:
#@title Plot the prediction for all points
# Plot the distribution for all points
fig = go.Figure()
fig.add_trace(go.Scatter(x=xgrid,y=f(xgrid),mode='lines',name="Originator function"))
fig.add_trace(go.Scatter(x=x,y=y,mode='markers',marker=dict(size=10),name="Generated data"))
#fig.add_trace(go.Scatter(x=[new_obs_point,new_obs_point],y=[new_obs_mu-2*new_obs_std,new_obs_mu+2*new_obs_std],mode='markers+lines',line=dict(color = px.colors.qualitative.G10[3]),name = "Conditional Deviation"))
fig.add_trace(go.Scatter(x=xgrid,y=xgrid_mu,mode='lines',line=dict(width=3,color = px.colors.qualitative.G10[3]),name = "Conditional Mean"))
fig.add_trace(go.Scatter(x=xgrid,y=np.array(xgrid_mu)-2*np.array(xgrid_std),mode='lines',line=dict(width=1,color = px.colors.qualitative.G10[3]),showlegend=False))
fig.add_trace(go.Scatter(x=xgrid,y=np.array(xgrid_mu)+2*np.array(xgrid_std),mode='lines',fill='tonexty',line=dict(width=1,color = px.colors.qualitative.G10[3]),name = "Conditional Deviation"))

fig.update_layout(showlegend=True,title="Conditional prediction at all points",xaxis_title="x",yaxis_title="y")
fig.show()

# Tasks

1. Run the same setup with a different kernel (uncomment the Laplacian or the polynomial kernel in `kernel_fcn`). How does the similarity between points influence the conditional probability?
2. Change the similarity scale `sigma` to 0.5 and to 5. What happens to the prediction between and outside the observation points?
3. Increase the measurement noise `NOISE_MEASUREMENTS` to 1. How does the uncertainty band change compared to the noise-free case?


In [45]:
# Your code here
